# Chapter 12: Custom Models and Training with TensorFlow

## A Quick Tour of TensorFlow
While Keras is a high-level API, TensorFlow provides the lower-level infrastructure. At its core, TensorFlow revolves around tensors, which flow from operation to operation.

<p align="left"><img src="../fig/figure12.1.png" width="45%"></p>

### Tensors and Operations

Tensors are multidimensional arrays, similar to NumPy arrays but they can hold scalars, strings, and other types.

- Tensors: Immutable objects.
- Operations: TF provides a vast library of math operations (add, multiply, matrix multiplication, etc.).

<p align="left"><img src="../fig/figure12.2.png" width="45%"></p>

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# Tensors and Operations
t = tf.constant([[1., 2., 3.], [4., 5., 6.]]) # A matrix
print(t.shape)
print(t.dtype)

# Indexing (like NumPy)
print(t[:, 1:])

# Operations
print(t + 10)
print(tf.square(t))
print(t @ tf.transpose(t)) # Matrix multiplication

(2, 3)
<dtype: 'float32'>
tf.Tensor(
[[2. 3.]
 [5. 6.]], shape=(2, 2), dtype=float32)
tf.Tensor(
[[11. 12. 13.]
 [14. 15. 16.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[ 1.  4.  9.]
 [16. 25. 36.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[14. 32.]
 [32. 77.]], shape=(2, 2), dtype=float32)


### Tensors and NumPy
TensorFlow plays nicely with NumPy. You can create tensors from NumPy arrays and vice versa. However, Type Conversions are not automatic in TensorFlow. You must cast types explicitly to avoid performance hits or errors (e.g., adding a float tensor to an integer tensor throws an error).

In [2]:
# Tensors and NumPy
a = np.array([2., 4., 5.])
t = tf.constant(a)
print(t.numpy()) # Convert back to NumPy

# Type Conversions
t2 = tf.constant(40., dtype=tf.float64)
# tf.constant(2.0) + tf.constant(40) # This would fail (float32 vs int32)
t3 = tf.constant(2.0) + tf.cast(t2, tf.float32) # Explicit cast needed

[2. 4. 5.]


## Custom Loss Functions
Sometimes standard loss functions (like MSE or Cross-Entropy) aren't enough. For example, the Huber Loss is less sensitive to outliers than MSE.

### Defining a Custom Loss
You can define a simple function that takes y_true and y_pred and returns the loss value for each instance.

In [3]:
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

# Compiling the model with the custom loss
model = keras.models.Sequential([keras.layers.Dense(1, input_shape=[8])])
model.compile(loss=huber_fn, optimizer="sgd")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Saving and Loading Models with Custom Objects
When saving a model containing a custom function, Keras saves the function name. To load it, you must map the name to the actual function using custom_objects.

In [4]:
model.save("my_model_with_a_custom_loss.h5")
model = keras.models.load_model("my_model_with_a_custom_loss.h5",
                                custom_objects={"huber_fn": huber_fn})

### Subclassing Loss for Parameters
If your loss function has hyperparameters (like the threshold in Huber Loss) that you want to save with the model, you should subclass keras.losses.Loss.

In [5]:
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss = self.threshold * tf.abs(error) - self.threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

# Usage
model.compile(loss=HuberLoss(2.0), optimizer="sgd")

## Custom Metrics
Metrics are similar to losses but are used for evaluation (human-readable) and are not used for training (gradients).

### Streaming Metrics
Metrics can be stateful (e.g., Accuracy accumulates counts over batches). To create a custom stateful metric (like a streaming Huber Metric), subclass keras.metrics.Metric. You need to implement update_state, result, and reset_states.

In [6]:
class HuberMetric(keras.metrics.Metric):
    def __init__(self, threshold=1.0, **kwargs):
        super().__init__(**kwargs) # handles base args (e.g., name)
        self.threshold = threshold
        self.huber_fn = create_huber(threshold) # Assuming helper exists or logic inline
        self.total = self.add_weight("total", initializer="zeros")
        self.count = self.add_weight("count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        metric = self.huber_fn(y_true, y_pred)
        self.total.assign_add(tf.reduce_sum(metric))
        self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.total / self.count

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

# Helper for the example above
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < threshold
        squared_loss = tf.square(error) / 2
        linear_loss = threshold * tf.abs(error) - threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    return huber_fn

## Custom Layers
If you need a layer that doesn't exist in Keras (e.g., a weird activation or pooling), you can create one.

### Stateless Layers (Lambda)
For layers with no learnable weights, use keras.layers.Lambda.

In [7]:
exponential_layer = keras.layers.Lambda(lambda x: tf.exp(x))

### Stateful Layers (Subclassing)
For layers with weights, subclass keras.layers.Layer.

- __init__: Save hyperparameters.
- build: Create weights (kernels/biases). Called automatically when input shape is known.
- call: Perform the forward computation.
- compute_output_shape: Optional, but useful.

In [8]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
            name="kernel", shape=[batch_input_shape[-1], self.units],
            initializer="glorot_normal")
        self.bias = self.add_weight(
            name="bias", shape=[self.units], initializer="zeros")
        super().build(batch_input_shape) # Must be at the end

    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)

    def compute_output_shape(self, batch_input_shape):
        return tf.TensorShape(batch_input_shape.as_list()[:-1] + [self.units])

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "units": self.units,
                "activation": keras.activations.serialize(self.activation)}

## Custom Models
For complex architectures (like ResNet with residuals and loops), you can subclass keras.Model. It works like a Custom Layer but includes extra functionality like compile(), fit(), and evaluate().

<p align="left"><img src="../fig/figure12.3.png" width="45%"></p>

In [9]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self, n_layers, n_neurons, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(n_neurons, activation="elu",
                                          kernel_initializer="he_normal")
                       for _ in range(n_layers)]

    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        return inputs + Z

class ResidualRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(30, activation="elu",
                                          kernel_initializer="he_normal")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = keras.layers.Dense(output_dim)

    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(1 + 3): # Just an example loop
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)

## Losses and Metrics Based on Model Internals
Sometimes the loss depends on the model's internals (e.g., regularization loss or reconstruction loss in Autoencoders), not just y_pred and y_true. You can calculate this inside the call() method and use add_loss().

In [10]:
class ReconstructingRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(30, activation="selu",
                                          kernel_initializer="lecun_normal")
                       for _ in range(5)]
        self.out = keras.layers.Dense(output_dim)
        # Reconstruction part
        self.reconstruction_mean = keras.metrics.Mean(name="reconstruction_error")

    def build(self, batch_input_shape):
        n_inputs = batch_input_shape[-1]
        self.reconstruct = keras.layers.Dense(n_inputs)
        super().build(batch_input_shape)

    def call(self, inputs, training=None):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        reconstruction = self.reconstruct(Z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))
        self.add_loss(0.05 * recon_loss) # Add auxiliary loss
        if training:
            result = self.reconstruction_mean(recon_loss)
            self.add_metric(result)
        return self.out(Z)

## Computing Gradients with Autodiff
To understand custom training loops, we need Automatic Differentiation. TensorFlow uses tf.GradientTape to record operations for gradient computation.

In [11]:
def f(w1, w2):
    return 3 * w1 ** 2 + 2 * w1 * w2

w1, w2 = tf.Variable(5.), tf.Variable(3.)
with tf.GradientTape() as tape:
    z = f(w1, w2)

gradients = tape.gradient(z, [w1, w2])
print(gradients) # [30.0, 10.0]

[<tf.Tensor: shape=(), dtype=float32, numpy=36.0>, <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]


## Custom Training Loops
If fit() is too limiting, you can write your own loop. This gives you total control over the training process.
1. Iterate over epochs.
2. Iterate over batches using tf.data.
3. Compute loss inside a GradientTape.
4. Compute gradients.
5. Apply gradients using the optimizer.
6. Update metrics.

In [16]:
# Setup
l2_reg = keras.regularizers.l2(0.05)
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal",
                       kernel_regularizer=l2_reg),
    keras.layers.Dense(1, kernel_regularizer=l2_reg)
])

def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

def print_status_bar(iteration, total, loss, metrics=None):
    metrics = " - ".join(["{}: {:.4f}".format(m.name, m.result())
                          for m in [loss] + (metrics or [])])
    end = "" if iteration < total else "\n"
    print("\r{}/{} - ".format(iteration, total) + metrics, end=end)

# Training Loop
n_epochs = 5
batch_size = 32
n_steps = 100 # Example steps
optimizer = keras.optimizers.SGD(learning_rate=0.01)
loss_fn = keras.losses.MeanSquaredError()
mean_loss = keras.metrics.Mean(name="mean_loss")
metrics = [keras.metrics.MeanAbsoluteError()]

# Create dummy data for example to run
X_train_scaled = np.random.randn(1000, 5).astype(np.float32)
y_train = np.random.randn(1000, 1).astype(np.float32)

for epoch in range(n_epochs):
    print("Epoch {}/{}".format(epoch + 1, n_epochs))
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train_scaled, y_train)
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses) # Add reg losses

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        # Constraints handling (if any)
        for variable in model.variables:
            if variable.constraint is not None:
                variable.assign(variable.constraint(variable))

        mean_loss(loss)
        for metric in metrics:
            metric(y_batch, y_pred)

        print_status_bar(step, n_steps, mean_loss, metrics)

    # Reset metrics at end of epoch
    print_status_bar(n_steps, n_steps, mean_loss, metrics)
    for metric in [mean_loss] + metrics:
        metric.reset_state()

Epoch 1/5
100/100 - mean_loss: 3.9646 - mean_absolute_error: 0.8751
100/100 - mean_loss: 3.9646 - mean_absolute_error: 0.8751
Epoch 2/5
100/100 - mean_loss: 3.3214 - mean_absolute_error: 0.8378
100/100 - mean_loss: 3.3214 - mean_absolute_error: 0.8378
Epoch 3/5
100/100 - mean_loss: 2.8412 - mean_absolute_error: 0.8052
100/100 - mean_loss: 2.8412 - mean_absolute_error: 0.8052
Epoch 4/5
100/100 - mean_loss: 2.5262 - mean_absolute_error: 0.8084
100/100 - mean_loss: 2.5262 - mean_absolute_error: 0.8084
Epoch 5/5
100/100 - mean_loss: 2.2604 - mean_absolute_error: 0.8206
100/100 - mean_loss: 2.2604 - mean_absolute_error: 0.8206


## TensorFlow Functions and Graphs
TensorFlow 2.0 uses Eager Execution by default (easier debugging). However, converting Python functions into TensorFlow Graphs can significantly boost performance. The @tf.function decorator does this automatically via AutoGraph.

In [17]:
@tf.function
def cube(x):
    return x ** 3

print(cube(tf.constant(2.0)))
print(cube(tf.constant([10., 20.])))

tf.Tensor(8.0, shape=(), dtype=float32)
tf.Tensor([1000. 8000.], shape=(2,), dtype=float32)
